In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error, r2_score

In [2]:
df = pd.read_csv("../data/raw/SBIN.csv")

df.head()

,Date,Adj Close,Close,High,Low,Open,Volume
0,2015-01-01,274.924988,314.000000,315.000000,310.700012,312.450012,6138488
1,2015-01-02,276.019440,315.250000,318.299988,314.350006,314.350006,9935094
2,2015-01-05,273.830566,312.750000,316.799988,312.100006,316.250000,9136716
3,2015-01-06,262.579651,299.899994,311.100006,298.700012,310.000000,15329257
4,2015-01-07,262.798492,300.149994,302.549988,295.149994,300.000000,15046745


In [3]:
df["Date"] = pd.to_datetime(df["Date"])

df = df.sort_values("Date").reset_index(drop=True)

print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2878 entries, 0 to 2877
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   Date       2878 non-null   datetime64[ns]
 1   Adj Close  2878 non-null   float64       
 2   Close      2878 non-null   float64       
 3   High       2878 non-null   float64       
 4   Low        2878 non-null   float64       
 5   Open       2878 non-null   float64       
 6   Volume     2878 non-null   int64         
dtypes: datetime64[ns](1), float64(5), int64(1)
memory usage: 157.5 KB
None


In [4]:
print("Start Date:", df["Date"].min())
print("End Date:", df["Date"].max())
print("Total Records:", len(df))

Start Date: 2015-01-01 00:00:00
End Date: 2026-08-24 00:00:00
Total Records: 2878


In [5]:
# 80% training, 20% testing

split_index = int(len(df) * 0.8)

train = df.iloc[:split_index].copy()
test = df.iloc[split_index:].copy()

print("Train shape:", train.shape)
print("Test shape:", test.shape)

print("\nTrain period:")
print(train["Date"].min(), "to", train["Date"].max())

print("\nTest period:")
print(test["Date"].min(), "to", test["Date"].max())

Train shape: (2302, 7)
Test shape: (576, 7)

Train period:
2015-01-01 00:00:00 to 2024-05-02 00:00:00

Test period:
2024-05-03 00:00:00 to 2026-08-24 00:00:00


In [6]:
# Get the Close prices

train_close = train["Close"].values
test_close = test["Close"].values

# For Naive Forecast:
# Prediction for a day = Previous day's actual closing price

naive_predictions = np.concatenate([
    [train_close[-1]],       # First test prediction = last train Close
    test_close[:-1]          # Remaining predictions = previous test Close
])

# Check first few predictions

comparison = pd.DataFrame({
    "Date": test["Date"].values[:10],
    "Actual_Close": test_close[:10],
    "Naive_Predicted": naive_predictions[:10]
})

comparison

,Date,Actual_Close,Naive_Predicted
0,2024-05-03,831.450012,830.049988
1,2024-05-06,807.799988,831.450012
2,2024-05-07,801.900024,807.799988
3,2024-05-08,810.799988,801.900024
4,2024-05-09,819.799988,810.799988
5,2024-05-10,817.349976,819.799988
6,2024-05-13,808.799988,817.349976
7,2024-05-14,818.200012,808.799988
8,2024-05-15,820.299988,818.200012
9,2024-05-16,811.950012,820.299988


In [7]:
# Calculate evaluation metrics

naive_mae = mean_absolute_error(test_close, naive_predictions)

naive_rmse = np.sqrt(
    mean_squared_error(test_close, naive_predictions)
)

naive_mape = mean_absolute_percentage_error(
    test_close,
    naive_predictions
) * 100

naive_r2 = r2_score(
    test_close,
    naive_predictions
)

print("===== NAIVE FORECAST RESULTS =====")

print(f"MAE: ₹{naive_mae:.2f}")
print(f"RMSE: ₹{naive_rmse:.2f}")
print(f"MAPE: {naive_mape:.2f}%")
print(f"R² Score: {naive_r2:.4f}")

===== NAIVE FORECAST RESULTS =====
MAE: ₹9.35
RMSE: ₹14.25
MAPE: 1.05%
R² Score: 0.9859
